# CrimeScope ML — 06 (UK). Export for Backend

Reads the latest scores / demographics / boundaries / lookup from Delta
and writes the JSON files that `crimescope/backend/app/data/` consumes,
landing them in the `exports/latest/` folder of the UC Volume so
`databricks fs cp` can pull them locally.

In [ ]:
spark.sql("USE CATALOG varanasi")
spark.sql("USE SCHEMA default")

In [ ]:
import os
import json
from datetime import datetime, timezone
from pyspark.sql import functions as F

VOLUME_OUT = "/Volumes/varanasi/default/ml_data_uk/exports/latest"
os.makedirs(VOLUME_OUT, exist_ok=True)


def write_records(table: str, fname: str, columns: list[str] | None = None) -> int:
    df = spark.table(table)
    if columns:
        df = df.select(*[c for c in columns if c in df.columns])
    pdf = df.toPandas()
    path = f"{VOLUME_OUT}/{fname}"
    pdf.to_json(path, orient="records", date_format="iso")
    print(f"  {fname}: {len(pdf):,} rows -> {os.path.getsize(path)/1_048_576:.2f} MB")
    return len(pdf)


# --- MSOA bundle (default UK city) ---
n_msoa_scores = write_records("varanasi.default.uk_msoa_risk_scores", "uk_msoa_risk_scores.json")
write_records("varanasi.default.uk_msoa_boundaries", "uk_msoa_boundaries.json",
              ["tract_geoid", "NAMELSAD", "wkt", "ALAND"])
# Demographics under the same `tract_geoid` key
msoa_demo = spark.table("varanasi.default.uk_msoa_demographics") \
    .selectExpr("msoa_code as tract_geoid",
                "total_pop as total_pop_acs",
                "imd_score",
                "imd_decile",
                "imd_income as poverty_rate_acs")
msoa_demo.toPandas().to_json(f"{VOLUME_OUT}/uk_msoa_demographics.json", orient="records")
print(f"  uk_msoa_demographics.json: {msoa_demo.count():,} rows")

# --- LSOA bundle (high-resolution) ---
n_lsoa_scores = write_records("varanasi.default.uk_lsoa_risk_scores", "uk_lsoa_risk_scores.json")
write_records("varanasi.default.uk_lsoa_boundaries", "uk_lsoa_boundaries.json",
              ["tract_geoid", "NAMELSAD", "wkt", "ALAND"])
lsoa_demo = spark.table("varanasi.default.uk_lsoa_demographics") \
    .selectExpr("lsoa_code as tract_geoid",
                "total_pop as total_pop_acs",
                "imd_score",
                "imd_decile",
                "imd_income as poverty_rate_acs")
lsoa_demo.toPandas().to_json(f"{VOLUME_OUT}/uk_lsoa_demographics.json", orient="records")
print(f"  uk_lsoa_demographics.json: {lsoa_demo.count():,} rows")

In [ ]:
# --- Pipeline stats — sourced from MLflow run + table counts ---
import mlflow
mlflow.set_registry_uri("databricks-uc")
client = mlflow.MlflowClient()

def latest_metric(model_name: str) -> tuple[str, dict]:
    try:
        mv = client.get_model_version_by_alias(model_name, "champion")
        run = client.get_run(mv.run_id)
        return mv.version, {k: v for k, v in run.data.metrics.items()}
    except Exception as e:  # noqa: BLE001
        print(f"  champion lookup failed for {model_name}: {e}")
        return "?", {}

v_lsoa, m_lsoa = latest_metric("varanasi.default.crimescope_uk_risk_model_lsoa")
v_msoa, m_msoa = latest_metric("varanasi.default.crimescope_uk_risk_model_msoa")

feat_stats = spark.sql("""
  SELECT COUNT(*) AS total_rows,
         COUNT(DISTINCT lsoa_code) AS n_regions,
         MIN(month_start) AS data_start,
         MAX(month_start) AS data_end
  FROM varanasi.default.uk_lsoa_features
""").first().asDict()

stats = [{
    "n_tracts": int(feat_stats["n_regions"]),
    "data_start": str(feat_stats["data_start"]),
    "data_end": str(feat_stats["data_end"]),
    "total_rows": int(feat_stats["total_rows"]),
    "model_lsoa_version": v_lsoa,
    "model_msoa_version": v_msoa,
    "model_lsoa_mae": float(m_lsoa.get("ensemble_mae", m_lsoa.get("log_mae", 0.0))),
    "model_msoa_mae": float(m_msoa.get("ensemble_mae", m_msoa.get("log_mae", 0.0))),
    "scope": "England & Wales (LSOA + MSOA 2021)",
    "boundary_source": "ONS Open Geography Portal — LSOA Dec 2021 BGC + MSOA Dec 2021 BSC",
    "score_source": "LightGBM ensemble trained on 60 months of data.police.uk + ONS Census 2021 + IMD/WIMD 2019",
    "exported_at": datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z"),
}]
with open(f"{VOLUME_OUT}/uk_pipeline_stats.json", "w") as f:
    json.dump(stats, f)
print(f"  uk_pipeline_stats.json: {stats[0]['n_tracts']:,} regions, model v{v_lsoa} (LSOA) / v{v_msoa} (MSOA)")

In [ ]:
print("\nAll UK exports landed in", VOLUME_OUT)
for f in sorted(os.listdir(VOLUME_OUT)):
    sz = os.path.getsize(f"{VOLUME_OUT}/{f}") / 1_048_576
    print(f"  {f:40s}  {sz:>6.2f} MB")